# 04 — Monte Carlo Simulation and Numerical Validation

Use independent simulated portfolios for male and female mortality schedules and benchmark the results against analytical moments. The original random seed, assumptions and simulation code are retained.

## Project setup

Run this notebook from the repository root or from the `notebooks/` directory. All paths are resolved to the repository root. This project uses the `src/` helper modules already present in your repository; the original report did not include their source code.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent
if not (project_root / "src").is_dir():
    raise FileNotFoundError(
        "Project root not found. Open the repository in VS Code and run the notebook from the repository root or notebooks/."
    )
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
report_results_dir = project_root / "results"
report_results_dir.mkdir(parents=True, exist_ok=True)
figures_dir = project_root / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
clean_path = project_root / "data" / "processed" / "ssa_mortality_clean.csv"
if not clean_path.is_file():
    raise FileNotFoundError(
        "Run notebook 01 first to generate data/processed/ssa_mortality_clean.csv"
    )
mortality = pd.read_csv(clean_path)
print("Loaded", len(mortality), "mortality records")

from src.life_math import term_death_probabilities, term_epv, portfolio_moments

entry_age = 40
term = 20
benefit = 100_000
interest_rate = 0.03

death_tables = {}
actuarial_results = []
for sex in ["Male", "Female"]:
    death_table, survival_prob = term_death_probabilities(
        mortality, sex=sex, entry_age=entry_age, term=term
    )
    epv = term_epv(
        death_table, benefit=benefit, interest_rate=interest_rate
    )
    death_tables[sex] = death_table
    actuarial_results.append({
        "Sex": sex,
        "20-year survival probability": survival_prob,
        "20-year death probability": 1 - survival_prob,
        "Expected present value ($)": epv,
    })
results_df = pd.DataFrame(actuarial_results)


## Monte Carlo portfolio summary

In [ ]:
from src.simulation import simulate_aggregate_claims

rng = np.random.default_rng(2026)

n_policies = 1_000
n_portfolios = 10_000

simulation_results = {}
simulation_summary = []

for sex in ["Male", "Female"]:

    death_table = death_tables[sex]

    term_survival = (
        1.0 - death_table["death_probability"].sum()
    )

    simulated = simulate_aggregate_claims(
        death_table=death_table,
        term_survival=term_survival,
        n_policies=n_policies,
        n_portfolios=n_portfolios,
        benefit=benefit,
        interest_rate=interest_rate,
        rng=rng
    )

    simulation_results[sex] = simulated

    epv = results_df.loc[
        results_df["Sex"] == sex,
        "Expected present value ($)"
    ].iloc[0]

    theoretical_mean = n_policies * epv

    simulated_mean = simulated.mean()
    simulated_sd = simulated.std(ddof=1)
    mcse = simulated_sd / np.sqrt(n_portfolios)

    simulation_summary.append({
        "Sex": sex,
        "Theoretical mean ($)": theoretical_mean,
        "Simulated mean ($)": simulated_mean,
        "Simulation SD ($)": simulated_sd,
        "Monte Carlo SE ($)": mcse,
        "95th percentile ($)": np.percentile(
            simulated, 95
        )
    })

simulation_df = pd.DataFrame(simulation_summary)

simulation_df["Coefficient of variation (%)"] = (
    simulation_df["Simulation SD ($)"]
    / simulation_df["Simulated mean ($)"]
    * 100
)

display(simulation_df.round(2))

## Aggregate discounted claims distributions

In [ ]:
for sex in ["Male", "Female"]:

    simulated = simulation_results[sex]

    theoretical_mean, _ = portfolio_moments(
        death_tables[sex],
        benefit,
        interest_rate,
        n_policies
    )

    percentile_95 = np.percentile(simulated, 95)

    fig, ax = plt.subplots(figsize=(10, 6))

    # Express claims in millions of dollars
    ax.hist(
        simulated / 1_000_000,
        bins=45,
        alpha=0.75,
        edgecolor="white"
    )

    ax.axvline(
        theoretical_mean / 1_000_000,
        linestyle="--",
        linewidth=2,
        label="Theoretical mean"
    )

    ax.axvline(
        percentile_95 / 1_000_000,
        linestyle=":",
        linewidth=2,
        label="95th percentile"
    )

    ax.set_title(
        f"Simulated Aggregate Claims — {sex} Mortality"
    )
    ax.set_xlabel(
        "Aggregate present value of claims ($ millions)"
    )
    ax.set_ylabel("Number of simulated portfolios")

    ax.legend()
    ax.grid(alpha=0.2)

    fig.tight_layout()

    fig.savefig(
        figures_dir /
        f"aggregate_claims_{sex.lower()}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

## Numerical validation

In [ ]:
from src.life_math import portfolio_moments

validation_results = []

for sex in ["Male", "Female"]:

    death_table = death_tables[sex]
    simulated = simulation_results[sex]

    theoretical_mean, theoretical_sd = portfolio_moments(
        death_table=death_table,
        benefit=benefit,
        interest_rate=interest_rate,
        n_policies=n_policies
    )

    simulated_mean = simulated.mean()
    simulated_sd = simulated.std(ddof=1)

    theoretical_mcse = (
        theoretical_sd / np.sqrt(n_portfolios)
    )

    standardized_error = (
        simulated_mean - theoretical_mean
    ) / theoretical_mcse

    validation_results.append({
        "Sex": sex,
        "Theoretical mean": theoretical_mean,
        "Simulated mean": simulated_mean,
        "Theoretical SD": theoretical_sd,
        "Simulated SD": simulated_sd,
        "Mean error (MCSE units)": standardized_error
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df.round(3))

## Export simulation and validation tables

In [ ]:
def write_report_table(frame, filename, *, include_index=False, floatfmt=None):
    """Write an output table that the code-free Quarto report can include."""
    kwargs = {"index": include_index}
    if floatfmt is not None:
        kwargs["floatfmt"] = floatfmt
    (report_results_dir / filename).write_text(
        frame.to_markdown(**kwargs) + "\n", encoding="utf-8"
    )

# Format the simulation summary as a three-column, reader-friendly table.
summary_for_report = (
    simulation_df
    .set_index("Sex")
    .T
    .rename(index={
        "Theoretical mean ($)": "Theoretical mean (USD)",
        "Simulated mean ($)": "Simulated mean (USD)",
        "Simulation SD ($)": "Simulation SD (USD)",
        "Monte Carlo SE ($)": "Monte Carlo SE (USD)",
        "95th percentile ($)": "95th percentile (USD)",
    })
    .rename_axis("Measure")
    .reset_index()
)

write_report_table(
    summary_for_report,
    "08_simulation_summary.qmd",
    floatfmt=",.2f"
)

# Format the numerical-validation table without altering any calculations.
validation_for_report = validation_df.rename(columns={
    "Theoretical mean": "Theoretical mean (USD)",
    "Simulated mean": "Simulated mean (USD)",
    "Theoretical SD": "Theoretical SD (USD)",
    "Simulated SD": "Simulated SD (USD)",
})

write_report_table(
    validation_for_report,
    "10_numerical_validation.qmd",
    floatfmt=("", ",.2f", ",.2f", ",.2f", ",.2f", ".3f"),
)
print("Updated simulation and validation tables; histogram figures saved above.")
